# Brent Oil Prices - Initial Exploratory Data Analysis (Task 1)

This notebook performs the initial EDA required for the Week 10 interim submission:

- Load and clean the daily Brent crude oil price series (1987-2022)
- Plot the raw price series and annotate curated key events
- Compute and plot daily **log returns** to assess stationarity and volatility clustering
- Examine **rolling volatility**
- Run **Augmented Dickey-Fuller (ADF)** stationarity tests on price vs. log returns

Reusable logic lives in `src/` and is imported here.

In [1]:
import sys
from pathlib import Path

# Ensure the project root is importable when running from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.data_loader import add_log_returns, load_events, load_prices, save_processed
from src import eda

pd.set_option("display.max_columns", None)

## 1. Load and clean the data

In [2]:
prices = add_log_returns(load_prices())
events = load_events()
save_processed(prices)
print(f"Rows: {len(prices):,}")
print(f"Date range: {prices['Date'].min().date()} to {prices['Date'].max().date()}")
prices.head()

Rows: 9,011
Date range: 1987-05-20 to 2022-11-14


,Date,Price,LogPrice,LogReturn
0,1987-05-20,18.63,2.924773,NaN
1,1987-05-21,18.45,2.915064,-0.009709
2,1987-05-22,18.55,2.920470,0.005405
3,1987-05-25,18.60,2.923162,0.002692
4,1987-05-26,18.63,2.924773,0.001612


In [3]:
prices[["Price", "LogReturn"]].describe()

,Price,LogReturn
count,9011.000000,9010.000000
mean,48.420782,0.000179
std,32.860110,0.025532
min,9.100000,-0.643699
25%,19.050000,-0.011154
50%,38.570000,0.000402
75%,70.090000,0.012127
max,143.950000,0.412023


## 2. Raw price series with key events

The dashed red lines mark the curated geopolitical / economic / OPEC events.

In [4]:
eda.plot_price_series(prices, events)

WindowsPath('C:/Users/USER/Documents/Projects/10 academy/brent-oil-change-point-analysis/reports/figures/price_series.png')

## 3. Daily log returns (volatility clustering)

In [5]:
eda.plot_log_returns(prices)

WindowsPath('C:/Users/USER/Documents/Projects/10 academy/brent-oil-change-point-analysis/reports/figures/log_returns.png')

In [6]:
eda.plot_rolling_volatility(prices)

WindowsPath('C:/Users/USER/Documents/Projects/10 academy/brent-oil-change-point-analysis/reports/figures/rolling_volatility.png')

In [7]:
eda.plot_return_distribution(prices)

WindowsPath('C:/Users/USER/Documents/Projects/10 academy/brent-oil-change-point-analysis/reports/figures/return_distribution.png')

## 4. Stationarity testing (Augmented Dickey-Fuller)

A p-value below 0.05 indicates the series is stationary. We expect the raw price
to be **non-stationary** and the log returns to be **stationary**, which motivates
modeling the change points on log returns.

In [8]:
adf_price = eda.adf_report(prices["Price"])
adf_ret = eda.adf_report(prices["LogReturn"])
print("ADF on Price:      p-value =", round(adf_price["p_value"], 4),
      "| stationary:", adf_price["stationary_at_5pct"])
print("ADF on LogReturn:  p-value =", round(adf_ret["p_value"], 4),
      "| stationary:", adf_ret["stationary_at_5pct"])

ADF on Price:      p-value = 0.2893 | stationary: False
ADF on LogReturn:  p-value = 0.0 | stationary: True


In [9]:
import json
print(json.dumps(eda.summarize(prices), indent=2))

{
  "n_obs": 9011,
  "start_date": "1987-05-20",
  "end_date": "2022-11-14",
  "min_price": 9.1,
  "max_price": 143.95,
  "mean_price": 48.42078237709466,
  "adf_price": {
    "adf_statistic": -1.9938560113924675,
    "p_value": 0.28927350489340287,
    "used_lag": 29,
    "n_obs": 8981,
    "critical_values": {
      "1%": -3.4310783342658615,
      "5%": -2.861861876398633,
      "10%": -2.566941329781918
    },
    "stationary_at_5pct": false
  },
  "adf_log_return": {
    "adf_statistic": -16.427113494485894,
    "p_value": 2.4985801611428892e-29,
    "used_lag": 28,
    "n_obs": 8981,
    "critical_values": {
      "1%": -3.4310783342658615,
      "5%": -2.861861876398633,
      "10%": -2.566941329781918
    },
    "stationary_at_5pct": true
  }
}


## 5. Initial findings

- The raw price series is dominated by large, persistent regime shifts: the 2008 spike to ~$147 and subsequent crash, the 2014-2016 collapse, and the 2020 COVID crash. These are candidate structural breaks for the change point model.
- Daily **log returns** fluctuate around zero and show clear **volatility clustering** - calm periods interrupted by bursts of high volatility around crisis events.
- The return distribution is sharply peaked with heavy tails (leptokurtic), typical of financial return series.
- **ADF tests** confirm the raw price is non-stationary while log returns are stationary, so the Bayesian change point model in Task 2 will be applied to log returns (or to the mean of the price with an explicit switch point).